### Main Function

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import OrderedDict

def jump_flight_time_py(df, plot=True):
    # build OrderedDict {timestamp: acceleration}
    valuesFixed = OrderedDict(zip(df['Timestamp:'].astype(int), df['Acceleration:'].astype(float)))

    values = valuesFixed
    while True:
        # keys [Timestamps], vals [Acceleration]
        keys, vals = list(values.keys()), list(values.values())
        #vals[i] is greater than its left neighbor (vals[i-1]) 
        #and greater than its right neighbor (vals[i+1])
        valuesMax = OrderedDict({keys[i]: vals[i] for i in range(1, len(vals)-1)
                                 if vals[i-1] < vals[i] > vals[i+1]})
        if len(valuesMax) >= 3:
            values = valuesMax
            continue
        break
    # invalid jump !!
    if len(values) < 3: 
        return None

    # take 3 max peaks from updated values
    vals, keys = list(values.values()), list(values.keys())
    # takes the last 3 values from the sorted list
    top3 = sorted(set(vals))[-3:]
    peaks = sorted([keys[vals.index(v)] for v in top3])
    t1, t2, t3 = peaks

    #fragments & local minima
    #frag1 contains the points between t1 and t2.
    #frag2 contains the points between t2 and t3.
    frag1 = {k:v for k,v in valuesFixed.items() if t1 <= k < t2}
    frag2 = {k:v for k,v in valuesFixed.items() if t2 <= k < t3}
    if not frag1 or not frag2: 
        return None

    #takes the smallest value in every frag.
    #takes the key (timestamp) associated with this smallest value.
    val1, t_val1 = min(frag1.values()), min(frag1, key=frag1.get)
    val2, t_val2 = min(frag2.values()), min(frag2, key=frag2.get)
    
    #The plot illustrates the detection of the 3 main peaks highlighted in red with their corresponding values. 
    #Between 3 peaks, the two local minima are also identified, the 1 minimum in green and the 2 in blue, 
    #the jump flight time is represented in orange, 
    #indicating the interval measured between the take-off and landing phases.
    if plot:
        t, acc = list(valuesFixed.keys()), list(valuesFixed.values())
        plt.figure(figsize=(10,5))
        
        # out of jump signal
        plt.plot(t, acc, color='lightgray', label="Acceleration")
        
        # jump signal
        t_jump = [k for k in t if t_val1 <= k <= t_val2]
        acc_jump = [valuesFixed[k] for k in t_jump]
        plt.plot(t_jump, acc_jump, color='orange', label="jump flight signal")
        
        # 3 maximum peaks (red)
        for tp in peaks:
            plt.plot(tp, valuesFixed[tp], 'ro', markersize=5)
            plt.text(tp, valuesFixed[tp]+0.05, f"({tp},{valuesFixed[tp]:.2f})", 
                     ha="center", color="red", fontsize=8)
        
        # minima
        plt.plot(t_val1, val1, 'go', markersize=6, label="val1 (min frag1)")
        plt.text(t_val1, val1-0.1, f"({t_val1},{val1:.2f})", ha="center", color="green", fontsize=8)
        plt.plot(t_val2, val2, 'bo', markersize=6, label="val2 (min frag2)")
        plt.text(t_val2, val2-0.1, f"({t_val2},{val2:.2f})", ha="center", color="blue", fontsize=8)
        
        plt.xlabel("Timestamp (ms)")
        plt.ylabel("Acceleration")
        plt.title("3 peaks detection (red) & minima (val1, val2) - jump time in orange")
        plt.legend()
        # Save the plot
        plt.savefig("jft.png", dpi=300, bbox_inches='tight')
        plt.show()   
        
    # return Time of flight 
    return abs(t_val2 - t_val1)
    
#first trial example
df_user1 = pd.read_csv("./TempoVooDataset/Mobile/1.txt", sep="\t")
aux1 = jump_flight_time_py(df_user1)
print(f"Jump Flight Time {aux1}ms" )

### Apply the function to Dataset and export the results to a CSV file

In [ ]:
import glob
import os

all_files = glob.glob("./TempoVooDataset/Mobile/*.txt")
results = []

for file_path in all_files:
    try:
        df = pd.read_csv(file_path, sep="\t")
        tof_aux = jump_flight_time_py(df, False) # put plot flag to True to show all 550 Trials plots
        user_id = os.path.basename(file_path).split(".")[0] #1.txt returned user_id will be 1
        results.append({"Number:": int(user_id), "time_of_fly_py": tof_aux}) # column of new file 
    except Exception as e:
        print(f"file problem {file_path}: {e}")

# Export to CSV
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Number:")  # sort by Number column
df_results.to_csv("./TempoVooDataset/time_of_fly_all_trials.csv", index=False) #export to csv
print("Export completed for all trials!")